<h5> Imports </h5>

In [1]:
from dotenv import load_dotenv
import os

from azure.identity import DefaultAzureCredential
from openai import AzureOpenAI

import fitz 
import numpy as np
from paddleocr import PaddleOCR

load_dotenv()

/Volumes/BackupHD/Projects/variant-logic/vl-blogs/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

<h5> Experimenting with PaddleOCR, drawback is that it is slow with documents with a high number of pages </h5>

In [2]:
def ocr_document(pdf_path: str, dpi: int = 300, lang: str = "en", conf_min: float = 0.5, native_min_chars: int = 40):
    """
    OCR a single PDF document (path).
    - Extracts native text if available.
    - Falls back to PaddleOCR for scanned pages.
    Returns: list of dicts [{page, method, text}]
    """
    # Initialize PaddleOCR with correct parameters for v3.x
    ocr = PaddleOCR(use_textline_orientation=True, lang=lang)

    def page_to_image(page):
        zoom = dpi / 72
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
        if pix.n == 4:
            img = img[:, :, :3]
        return img

    doc = fitz.open(pdf_path)
    pages = []

    for i, page in enumerate(doc, start=1):
        native_text = (page.get_text("text") or "").strip()

        if len(native_text) >= native_min_chars:
            text = native_text
            method = "native"
        else:
            img = page_to_image(page)
            result = ocr.predict(img)
            lines = []
            if result and result[0]:
                for line in result[0]:
                    if line and len(line) >= 2:
                        # Structure: [[bbox], (text, confidence)]
                        txt, conf = line[1][0], line[1][1]
                        if conf >= conf_min:
                            lines.append(txt.strip())
            text = "\n".join(lines).strip()
            method = "ocr"

        pages.append({"page": i, "method": method, "text": text})

    doc.close()
    return pages

<h5> Since we just need a text, plain pdf to text should do the job. </h5>

In [3]:
def extract_pdf_text_simple(pdf_path: str):
    """
    Extract text from PDF using native text extraction only.
    Returns: list of dicts [{page, text}]
    """
    doc = fitz.open(pdf_path)
    pages = []
    
    for i, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()
        pages.append({
            "page": i,
            "text": text,
            "method": "mypdf"
        })
    
    doc.close()
    return pages

In [ ]:
data = extract_pdf_text_simple("data/fortimo-led-linear-dig.pdf")
#data = ocr_document("data/zgp-stand-alone-ip65-sensors.pdf", lang="en")

In [ ]:
# Format and save extracted text to a file

output_dir = "data_output"
os.makedirs(output_dir, exist_ok=True)

output_file = os.path.join(output_dir, "extracted_text.txt")

with open(output_file, "w", encoding="utf-8") as f:
    for page_data in data:
        f.write(f"{'='*80}\n")
        f.write(f"PAGE {page_data['page']} (Extraction method: {page_data['method']})\n")
        f.write(f"{'='*80}\n\n")
        f.write(page_data['text'])
        f.write(f"\n\n")

print(f"✓ Extracted text saved to: {output_file}")
print(f"  Total pages: {len(data)}")
print(f"  Total characters: {sum(len(p['text']) for p in data)}")

In [ ]:
# Open the file in VS Code
from IPython.display import display, FileLink

# Display clickable link
display(FileLink(output_file))

In [ ]:
azure_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION")
)

def normalize(v):
    """ Normalize a vector to unit length """
    v = np.array(v, dtype="float32")
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v


def get_text_embedding(text, normalizeEmbedding=True, model="text-embedding-3-large"):
    """ Get text embedding from Azure OpenAI and optionally normalize it """
    emb = azure_client.embeddings.create(input=[text], model=model).data[0].embedding
    if normalizeEmbedding:
        emb = normalize(emb)
    return emb

In [5]:
import glob

# Process all PDFs in data folder

all_results = []

pdf_files = glob.glob("data/*.pdf")

print(f"Found {len(pdf_files)} PDF files")

for pdf_path in pdf_files:
    try:
        print(f"Processing: {pdf_path}: {pdf_files.index(pdf_path)+1}/{len(pdf_files)}")
        pages = extract_pdf_text_simple(pdf_path)
        all_results.append({
            "file": pdf_path,
            "pages": pages
        })
        print(f"  ✓ Extracted {len(pages)} pages")
    except Exception as e:
        print(f"  ✗ Error processing {pdf_path}: {e}")

print(f"\n✓ Total files processed: {len(all_results)}")

Found 1069 PDF files
Processing: data/Xi_LP_200W_0.3-1.05A_S1_WL_I195_929002879180.pdf: 1/1069
  ✓ Extracted 8 pages
Processing: data/fortimo-led-linear-dig.pdf: 2/1069
  ✓ Extracted 60 pages
Processing: data/fortimo-slm-c-1203-L09-1619-g7-he.pdf: 3/1069
  ✓ Extracted 9 pages
Processing: data/Fortimo_LED_Strip_OC_2ft_2200lm_9xx_HV5_.pdf: 4/1069
  ✓ Extracted 8 pages
Processing: data/Xitanium_42W_a_0.9_1.05A_40V_3CG_230V_929001476706.pdf: 5/1069
  ✓ Extracted 7 pages
Processing: data/Xitanium_40W_R_0.3-1.05A_54V_TD_I_929002801206_2024-04-16.pdf: 6/1069
  ✓ Extracted 8 pages
Processing: data/Xitanium_60W_0.08-0.35A_300V_TD16_230V_929001681506.pdf: 7/1069
  ✓ Extracted 8 pages
Processing: data/Xi_42W_a_0.9-1.05A_40V_DS_3CB_230V_929002850280.pdf: 8/1069
  ✓ Extracted 7 pages
Processing: data/CertaDrive_60W_0.7A_85V_230V_929001613306.pdf: 9/1069
  ✓ Extracted 7 pages
Processing: data/Fortimo-SLM-C-8xx-1206-L15-1919-G8N-HE-2024-10-11.pdf: 10/1069
  ✓ Extracted 10 pages
Processing: data/Disma

In [8]:
get_text_embedding(all_results[0]['pages'][0]['text'],normalizeEmbedding=True)

BadRequestError: Error code: 400 - {'error': {'code': 'Tenant provided in token does not match resource token', 'message': 'Token tenant c970182f-4161-459c-8a07-2b00ed24c146 does not match resource tenant.'}}

In [6]:
# Calculate embeddings for each page and extend all_results

print("Calculating embeddings for all pages...")

for doc_result in all_results:
    file_name = doc_result["file"]
    pages = doc_result["pages"]
    
    print(f"Processing embeddings for: {file_name}")
    
    page_embeddings = []
    
    for page_data in pages:
        text = page_data["text"]
        
        if text.strip():
            try:
                embedding = get_text_embedding(text, normalizeEmbedding=True)
                page_embeddings.append(embedding)
                print(f"  ✓ Page {page_data['page']}: embedding calculated ({len(embedding)} dims)")
            except Exception as e:
                print(f"  ✗ Page {page_data['page']}: error - {e}")
                page_embeddings.append(None)
        else:
            print(f"  ⊘ Page {page_data['page']}: empty text, skipping")
    
    doc_result["embeddings"] = page_embeddings

print(f"\n✓ All embeddings calculated")
print(f"  Total documents: {len(all_results)}")
print(f"  Total embeddings: {sum(len([e for e in doc['embeddings'] if e is not None]) for doc in all_results)}")

Calculating embeddings for all pages...
Processing embeddings for: data/Xi_LP_200W_0.3-1.05A_S1_WL_I195_929002879180.pdf
  ✗ Page 1: error - Error code: 400 - {'error': {'code': 'Tenant provided in token does not match resource token', 'message': 'Token tenant c970182f-4161-459c-8a07-2b00ed24c146 does not match resource tenant.'}}
  ✗ Page 2: error - Error code: 400
  ✗ Page 3: error - Error code: 400
  ✗ Page 4: error - Error code: 400
  ✗ Page 5: error - Error code: 400
  ✗ Page 6: error - Error code: 400
  ✗ Page 7: error - Error code: 400
  ✗ Page 8: error - Error code: 400
Processing embeddings for: data/fortimo-led-linear-dig.pdf
  ✗ Page 1: error - Error code: 400
  ✗ Page 2: error - Error code: 400
  ✗ Page 3: error - Error code: 400
  ✗ Page 4: error - Error code: 400
  ✗ Page 5: error - Error code: 400
  ✗ Page 6: error - Error code: 400
  ✗ Page 7: error - Error code: 400
  ✗ Page 8: error - Error code: 400
  ✗ Page 9: error - Error code: 400
  ✗ Page 10: error - Error code:

KeyboardInterrupt: 